# 📊 Eval Metrics Deep Dive
## BLEU · ROUGE · BERTScore — From Scratch + Human Judgment Failures

---

### What you'll build & learn

| Metric | Core idea | Key weakness |
|--------|-----------|-------------|
| **BLEU** | n-gram precision + brevity penalty | Ignores recall; order-insensitive beyond n-gram window |
| **ROUGE** | n-gram / LCS recall-focused F1 | Surface-form only; synonyms invisible |
| **BERTScore** | Contextual embedding cosine similarity | Expensive; biased toward fluent-sounding noise |

All three are implemented **from scratch** — no `sacrebleu`, `evaluate`, or `rouge_score` libraries.

---

## 0 · Imports & Setup

In [ ]:
# Standard-library + lightweight deps only
import re
import math
import collections
from itertools import chain
from typing import List, Tuple, Dict, Optional

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# BERTScore needs a transformer model — we use sentence-transformers (tiny model)
try:
    from sentence_transformers import SentenceTransformer
    BERT_AVAILABLE = True
    print('✅ sentence-transformers found')
except ImportError:
    BERT_AVAILABLE = False
    print('⚠️  sentence-transformers not installed.')
    print('   Run: pip install sentence-transformers')
    print('   BERTScore cells will use a word-vector fallback for demo purposes.')

print('\n🔧 All core metrics will run regardless — BERTScore gracefully degrades.')

---
## PART 1 · BLEU Score — From Scratch

### 1.1 Theory

**BLEU** (Bilingual Evaluation Understudy, Papineni et al. 2002) measures **modified n-gram precision**.

$$\text{BLEU} = BP \cdot \exp\!\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

Where:
- $p_n$ = **modified n-gram precision** for order $n$  
- $w_n$ = weight per n-gram level (uniform = $1/N$)  
- $BP$ = **brevity penalty** = $\min(1,\; e^{1 - r/c})$, where $r$ = reference length, $c$ = candidate length

**Modified precision** clips each candidate n-gram count to the *maximum* count it appears in *any* reference, preventing reward for repetition.

### Key design decisions
1. Corpus-level (default) vs sentence-level (add +1 smoothing)
2. Geometric mean over n-gram orders: one zero $p_n$ kills the whole score
3. Brevity penalty only punishes short candidates, not long ones

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  BLEU — complete from-scratch implementation
# ─────────────────────────────────────────────────────────────────

def tokenize(text: str) -> List[str]:
    """Lowercase, punctuation-aware tokenizer (no NLTK)."""
    text = text.lower()
    # separate punctuation from words
    text = re.sub(r"([^\w\s'])", r" \1 ", text)
    return text.split()


def get_ngrams(tokens: List[str], n: int) -> collections.Counter:
    """Return a Counter of all n-grams in a token list."""
    return collections.Counter(
        tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)
    )


def modified_precision(
    hypothesis: List[str],
    references: List[List[str]],
    n: int
) -> Tuple[int, int]:
    """
    Compute clipped n-gram counts.
    Returns (numerator, denominator) to allow corpus-level accumulation.
    """
    hyp_ngrams = get_ngrams(hypothesis, n)
    if not hyp_ngrams:
        return 0, 0

    # Maximum reference count for each n-gram
    max_ref_counts: Dict[tuple, int] = {}
    for ref in references:
        ref_ngrams = get_ngrams(ref, n)
        for ng, cnt in ref_ngrams.items():
            max_ref_counts[ng] = max(max_ref_counts.get(ng, 0), cnt)

    # Clip hypothesis counts
    clipped = sum(
        min(cnt, max_ref_counts.get(ng, 0))
        for ng, cnt in hyp_ngrams.items()
    )
    total = sum(hyp_ngrams.values())
    return clipped, total


def brevity_penalty(hypothesis_len: int, reference_len: int) -> float:
    """BP = 1 if hyp longer than ref, else exp(1 - ref/hyp)."""
    if hypothesis_len >= reference_len:
        return 1.0
    if hypothesis_len == 0:
        return 0.0
    return math.exp(1 - reference_len / hypothesis_len)


def closest_reference_length(
    hypothesis_len: int,
    references: List[List[str]]
) -> int:
    """Ref length closest in absolute diff to hypothesis (tie → shorter)."""
    return min(
        (len(r) for r in references),
        key=lambda rl: (abs(rl - hypothesis_len), rl)
    )


def bleu_score(
    hypothesis: str,
    references: List[str],
    max_n: int = 4,
    weights: Optional[List[float]] = None,
    smoothing: bool = True
) -> Dict:
    """
    Sentence-level BLEU with optional +1 smoothing (Chen & Cherry 2014, method 1).

    Returns dict with:
      bleu        - final score (0-1)
      precisions  - [p1, p2, p3, p4]
      bp          - brevity penalty
      hyp_len     - hypothesis token count
      ref_len     - closest reference token count
    """
    if weights is None:
        weights = [1/max_n] * max_n

    hyp_tokens = tokenize(hypothesis)
    ref_tokens_list = [tokenize(r) for r in references]

    ref_len = closest_reference_length(len(hyp_tokens), ref_tokens_list)
    bp = brevity_penalty(len(hyp_tokens), ref_len)

    precisions = []
    for n in range(1, max_n + 1):
        num, den = modified_precision(hyp_tokens, ref_tokens_list, n)
        if smoothing:
            num += 1
            den += 1
        if den == 0:
            precisions.append(0.0)
        else:
            precisions.append(num / den)

    # Geometric mean in log space
    log_avg = sum(
        w * math.log(p) if p > 0 else float('-inf')
        for w, p in zip(weights, precisions)
    )
    score = bp * math.exp(log_avg) if log_avg != float('-inf') else 0.0

    return {
        'bleu': score,
        'precisions': precisions,
        'bp': bp,
        'hyp_len': len(hyp_tokens),
        'ref_len': ref_len
    }


# ── Quick sanity check ──────────────────────────────────────────
ref  = "The cat sat on the mat"
hyp1 = "The cat sat on the mat"       # perfect
hyp2 = "The cat is sitting on a mat"   # paraphrase
hyp3 = "A dog lay under a rug"         # wrong meaning

for label, hyp in [("Perfect match", hyp1), ("Paraphrase", hyp2), ("Wrong meaning", hyp3)]:
    r = bleu_score(hyp, [ref])
    print(f"{label:20s} → BLEU={r['bleu']:.4f}  P1={r['precisions'][0]:.3f}  "
          f"P2={r['precisions'][1]:.3f}  BP={r['bp']:.3f}")

### 1.2 Visualise BLEU internals

In [ ]:
# Visualise how each n-gram precision contributes
examples = {
    'Perfect': ('The cat sat on the mat', 'The cat sat on the mat'),
    'Good paraphrase': ('The cat sat on the mat', 'The cat rested on the mat'),
    'Word-order swap': ('The cat sat on the mat', 'On the mat sat the cat'),
    'Synonym': ('The cat sat on the mat', 'The feline rested on the rug'),
    'Short hypothesis': ('The cat', 'The cat sat on the mat'),
    'Repetition': ('The the the the the the', 'The cat sat on the mat'),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, (label, (ref_s, hyp_s)) in zip(axes, examples.items()):
    r = bleu_score(hyp_s, [ref_s])
    bars = ax.bar(['P1','P2','P3','P4'], r['precisions'],
                  color=['#4C72B0','#DD8452','#55A868','#C44E52'])
    ax.axhline(r['bleu'], color='black', linestyle='--', linewidth=1.5,
               label=f"BLEU={r['bleu']:.3f}  BP={r['bp']:.2f}")
    ax.set_title(label, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Precision')
    ax.legend(fontsize=8)
    ax.set_xlabel(f"HYP: \"{hyp_s[:35]}...\"" if len(hyp_s) > 35 else f"HYP: \"{hyp_s}\"")

plt.suptitle('BLEU: Per-order Precision Breakdown', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## PART 2 · ROUGE Score — From Scratch

### 2.1 Theory

**ROUGE** (Lin 2004) measures **recall**-oriented overlap. Three main variants:

| Variant | What it counts | Formula |
|---------|---------------|--------|
| ROUGE-N | n-gram overlap | $F_1$ of n-gram recall & precision |
| ROUGE-L | Longest Common Subsequence | LCS-based $F_1$ |
| ROUGE-S | Skip-bigrams | Pairs of words in order with gaps allowed |

$$\text{ROUGE-N} = \frac{\sum_{s \in \text{Refs}} \sum_{n\text{-gram} \in s} \text{Count}_{\text{match}}}{\sum_{s \in \text{Refs}} \sum_{n\text{-gram} \in s} \text{Count}}$$

The $F_1$ version (used in practice):
$$F_1 = \frac{2 \cdot P \cdot R}{P + R}$$

**LCS** for ROUGE-L captures in-order word matches without requiring contiguous windows.

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  ROUGE — complete from-scratch implementation
# ─────────────────────────────────────────────────────────────────

def rouge_n(
    hypothesis: str,
    reference: str,
    n: int = 2
) -> Dict[str, float]:
    """
    ROUGE-N precision, recall, and F1.
    """
    hyp_tokens = tokenize(hypothesis)
    ref_tokens = tokenize(reference)

    hyp_ngrams = get_ngrams(hyp_tokens, n)
    ref_ngrams = get_ngrams(ref_tokens, n)

    # Overlap = clipped to min of both counts
    overlap = sum(
        min(cnt, ref_ngrams[ng])
        for ng, cnt in hyp_ngrams.items()
    )

    hyp_total = sum(hyp_ngrams.values())
    ref_total = sum(ref_ngrams.values())

    precision = overlap / hyp_total if hyp_total else 0.0
    recall    = overlap / ref_total  if ref_total  else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)

    return {'precision': precision, 'recall': recall, 'f1': f1}


def lcs_length(x: List[str], y: List[str]) -> int:
    """
    Classic DP LCS (O(mn) time). Returns length of LCS.
    """
    m, n = len(x), len(y)
    # Use two rows to save memory
    prev = [0] * (n + 1)
    for i in range(1, m + 1):
        curr = [0] * (n + 1)
        for j in range(1, n + 1):
            if x[i-1] == y[j-1]:
                curr[j] = prev[j-1] + 1
            else:
                curr[j] = max(prev[j], curr[j-1])
        prev = curr
    return prev[n]


def rouge_l(
    hypothesis: str,
    reference: str,
    beta: float = 1.0
) -> Dict[str, float]:
    """
    ROUGE-L using LCS. Beta controls P/R trade-off (1 = F1).
    """
    hyp_tokens = tokenize(hypothesis)
    ref_tokens = tokenize(reference)

    lcs = lcs_length(hyp_tokens, ref_tokens)
    precision = lcs / len(hyp_tokens) if hyp_tokens else 0.0
    recall    = lcs / len(ref_tokens)  if ref_tokens  else 0.0

    denom = (1 + beta**2) * precision * recall
    numer = beta**2 * precision + recall
    f1 = denom / numer if numer > 0 else 0.0

    return {'precision': precision, 'recall': recall, 'f1': f1, 'lcs_length': lcs}


def skip_bigrams(tokens: List[str], k: int = None) -> collections.Counter:
    """
    Generate all skip-bigrams (ordered pairs of words with gap ≤ k).
    k=None means unlimited (all ordered pairs).
    """
    pairs = collections.Counter()
    for i in range(len(tokens)):
        j_end = len(tokens) if k is None else min(i + k + 2, len(tokens))
        for j in range(i + 1, j_end):
            pairs[(tokens[i], tokens[j])] += 1
    return pairs


def rouge_s(
    hypothesis: str,
    reference: str,
    skip_distance: int = None
) -> Dict[str, float]:
    """
    ROUGE-S (skip-bigram) F1.
    """
    hyp_tokens = tokenize(hypothesis)
    ref_tokens = tokenize(reference)

    hyp_sb = skip_bigrams(hyp_tokens, skip_distance)
    ref_sb = skip_bigrams(ref_tokens, skip_distance)

    overlap = sum(min(cnt, ref_sb[sb]) for sb, cnt in hyp_sb.items())

    precision = overlap / sum(hyp_sb.values()) if hyp_sb else 0.0
    recall    = overlap / sum(ref_sb.values())  if ref_sb  else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)

    return {'precision': precision, 'recall': recall, 'f1': f1}


def rouge_all(hypothesis: str, reference: str) -> Dict:
    """Convenience: compute ROUGE-1, ROUGE-2, ROUGE-L, ROUGE-S in one call."""
    return {
        'rouge1': rouge_n(hypothesis, reference, n=1),
        'rouge2': rouge_n(hypothesis, reference, n=2),
        'rougeL': rouge_l(hypothesis, reference),
        'rougeS': rouge_s(hypothesis, reference),
    }


# ── Sanity check ────────────────────────────────────────────────
reference  = "The cat sat on the mat"
hypotheses = [
    ("Perfect",          "The cat sat on the mat"),
    ("Synonym",          "The feline rested on the rug"),
    ("Order swapped",    "On the mat sat the cat"),
    ("Extra words",      "The large fluffy cat sat on the soft mat today"),
]

print(f"{'Label':20s}  R1-F1   R2-F1   RL-F1   RS-F1")
print("-" * 60)
for label, hyp in hypotheses:
    r = rouge_all(hyp, reference)
    print(f"{label:20s}  {r['rouge1']['f1']:.3f}   {r['rouge2']['f1']:.3f}   "
          f"{r['rougeL']['f1']:.3f}   {r['rougeS']['f1']:.3f}")

### 2.2 Visualise ROUGE variants

In [ ]:
examples_rouge = [
    ("Perfect",       "The cat sat on the mat", "The cat sat on the mat"),
    ("Good paraphrase","The cat sat on the mat", "The cat rested on the mat"),
    ("Synonym",        "The cat sat on the mat", "The feline rested on the rug"),
    ("Order swapped",  "The cat sat on the mat", "On the mat sat the cat"),
    ("High recall",    "The cat sat on the mat", "The big cat sat right on the mat yesterday"),
    ("Low recall",     "The cat sat on the mat", "A dog"),
]

variants = ['rouge1', 'rouge2', 'rougeL', 'rougeS']
labels   = [e[0] for e in examples_rouge]

scores = np.array([
    [rouge_all(hyp, ref)[v]['f1'] for v in variants]
    for ref, hyp, _ in [(e[1], e[2], None) for e in examples_rouge]
])

# wait — unpack correctly
scores = np.array([
    [rouge_all(e[2], e[1])[v]['f1'] for v in variants]
    for e in examples_rouge
])

x = np.arange(len(labels))
width = 0.2
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(14, 5))
for i, (variant, color) in enumerate(zip(variants, colors)):
    ax.bar(x + i*width - 1.5*width, scores[:, i], width, label=variant.upper(), color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('ROUGE Variant Comparison (F1)', fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

---
## PART 3 · BERTScore — From Scratch

### 3.1 Theory

**BERTScore** (Zhang et al. 2020) replaces n-gram matching with **contextual embedding similarity**.

1. Embed each token in hypothesis $\hat{x}$ and reference $x$ using a pre-trained transformer
2. For each hypothesis token, find the **maximum cosine similarity** to any reference token → **precision**
3. For each reference token, find the **maximum cosine similarity** to any hypothesis token → **recall**
4. Compute $F_1$

$$P_{\text{BERT}} = \frac{1}{|\hat{x}|} \sum_{\hat{x}_i \in \hat{x}} \max_{x_j \in x} \text{cos}(\hat{x}_i, x_j)$$
$$R_{\text{BERT}} = \frac{1}{|x|} \sum_{x_j \in x} \max_{\hat{x}_i \in \hat{x}} \text{cos}(x_j, \hat{x}_i)$$

**IDF weighting** (optional): rare tokens get higher weight, suppressing stop-word dominance.

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  BERTScore — from-scratch implementation
#  Uses sentence-transformers for embeddings (or fallback)
# ─────────────────────────────────────────────────────────────────

def cosine_similarity_matrix(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """
    A: (m, d), B: (n, d)
    Returns: (m, n) cosine similarity matrix
    """
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-9)
    return A_norm @ B_norm.T


def bertscore_from_embeddings(
    hyp_embeddings: np.ndarray,
    ref_embeddings: np.ndarray,
    idf_hyp: Optional[np.ndarray] = None,
    idf_ref: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    """
    Core BERTScore computation given pre-computed token embeddings.
    """
    sim = cosine_similarity_matrix(hyp_embeddings, ref_embeddings)  # (m, n)

    # Precision: each hyp token matched to best ref token
    p_scores = sim.max(axis=1)  # (m,)
    # Recall: each ref token matched to best hyp token
    r_scores = sim.max(axis=0)  # (n,)

    # IDF weighting
    if idf_hyp is not None:
        p_val = np.sum(p_scores * idf_hyp) / (np.sum(idf_hyp) + 1e-9)
    else:
        p_val = p_scores.mean()

    if idf_ref is not None:
        r_val = np.sum(r_scores * idf_ref) / (np.sum(idf_ref) + 1e-9)
    else:
        r_val = r_scores.mean()

    f1 = (2 * p_val * r_val / (p_val + r_val)
          if (p_val + r_val) > 0 else 0.0)

    return {'precision': float(p_val), 'recall': float(r_val), 'f1': float(f1),
            'sim_matrix': sim}


class BERTScorer:
    """
    BERTScore wrapper.
    If sentence-transformers is available, uses 'all-MiniLM-L6-v2' for
    sentence-level embeddings then splits by token.
    Falls back to random-projection word embeddings for CPU-only demos.
    """
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.use_bert = BERT_AVAILABLE
        if self.use_bert:
            print(f'Loading model: {model_name} ...')
            self.model = SentenceTransformer(model_name)
            print('Done.')
        else:
            print('Using deterministic hash-based fallback embeddings (demo only).')

    def _fallback_embed(self, tokens: List[str], dim: int = 64) -> np.ndarray:
        """Deterministic per-token embedding via hashing (for illustration only)."""
        rng = lambda s: np.array([
            math.sin(hash(s + str(i)) % 1000) for i in range(dim)
        ], dtype=float)
        return np.array([rng(t) for t in tokens])

    def embed_tokens(self, tokens: List[str]) -> np.ndarray:
        if self.use_bert:
            return self.model.encode(tokens, show_progress_bar=False)
        return self._fallback_embed(tokens)

    def score(self, hypothesis: str, reference: str) -> Dict[str, float]:
        hyp_tokens = tokenize(hypothesis)
        ref_tokens = tokenize(reference)

        if not hyp_tokens or not ref_tokens:
            return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

        hyp_emb = self.embed_tokens(hyp_tokens)
        ref_emb = self.embed_tokens(ref_tokens)

        result = bertscore_from_embeddings(hyp_emb, ref_emb)
        result['hyp_tokens'] = hyp_tokens
        result['ref_tokens'] = ref_tokens
        return result


# Instantiate (downloads ~90 MB the first time if using transformers)
scorer = BERTScorer()

In [ ]:
# ── Quick test ──────────────────────────────────────────────────
reference  = "The cat sat on the mat"
hypotheses = [
    ("Perfect",       "The cat sat on the mat"),
    ("Synonym",       "The feline rested on the rug"),
    ("Order swapped", "On the mat sat the cat"),
    ("Wrong meaning", "A dog lay under a rug"),
]

print(f"{'Label':20s}  BERTScore-P  BERTScore-R  BERTScore-F1")
print("-" * 60)
for label, hyp in hypotheses:
    r = scorer.score(hyp, reference)
    print(f"{label:20s}  {r['precision']:.4f}       {r['recall']:.4f}       {r['f1']:.4f}")

### 3.2 Visualise the similarity matrix

In [ ]:
def plot_bertscore_matrix(hypothesis: str, reference: str, title: str = ''):
    result = scorer.score(hypothesis, reference)
    sim = result['sim_matrix']
    hyp_tok = result['hyp_tokens']
    ref_tok = result['ref_tokens']

    fig, ax = plt.subplots(figsize=(max(6, len(ref_tok)*0.8), max(4, len(hyp_tok)*0.7)))
    im = ax.imshow(sim, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(ref_tok)))
    ax.set_xticklabels(ref_tok, rotation=45, ha='right')
    ax.set_yticks(range(len(hyp_tok)))
    ax.set_yticklabels(hyp_tok)
    ax.set_xlabel('Reference tokens')
    ax.set_ylabel('Hypothesis tokens')

    for i in range(len(hyp_tok)):
        for j in range(len(ref_tok)):
            ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center',
                    fontsize=8, color='white' if sim[i,j] > 0.6 else 'black')

    plt.colorbar(im, ax=ax)
    ax.set_title(f'{title}\nP={result["precision"]:.3f}  R={result["recall"]:.3f}  F1={result["f1"]:.3f}',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()


ref_s  = "The cat sat on the mat"
plot_bertscore_matrix("The feline rested on the rug", ref_s, title='BERTScore: Synonym case')
plot_bertscore_matrix("A dog lay under a table", ref_s, title='BERTScore: Different meaning')

---
## PART 4 · Where All Three Metrics Go Wrong

This is the most important section. We deliberately construct cases where **metrics disagree with human judgment** — high metric score but bad output, or low metric score but acceptable output.

Each cell defines:
- The reference
- A "good" and "bad" candidate
- Expected human ranking
- Observed metric ranking
- Explanation of *why* the metric fails

In [ ]:
def evaluate_pair(
    reference: str,
    hyp_A: str,
    hyp_B: str,
    label_A: str = 'Candidate A',
    label_B: str = 'Candidate B',
    case_title: str = ''
) -> None:
    """Print a full metric comparison for two hypotheses vs one reference."""
    def fmt(d): return f"BLEU={d['bleu']:.3f}  R1={d['r1']:.3f}  RL={d['rl']:.3f}  BS={d['bs']:.3f}"

    results = {}
    for label, hyp in [(label_A, hyp_A), (label_B, hyp_B)]:
        b  = bleu_score(hyp, [reference])['bleu']
        r1 = rouge_all(hyp, reference)['rouge1']['f1']
        rl = rouge_all(hyp, reference)['rougeL']['f1']
        bs = scorer.score(hyp, reference)['f1']
        results[label] = {'bleu': b, 'r1': r1, 'rl': rl, 'bs': bs}

    print(f"\n{'='*70}")
    print(f"  FAILURE CASE: {case_title}")
    print(f"{'='*70}")
    print(f"  REF : {reference}")
    print(f"  {label_A:10s}: {hyp_A}")
    print(f"  {label_B:10s}: {hyp_B}")
    print()
    for label in [label_A, label_B]:
        d = results[label]
        print(f"  {label:10s} → {fmt(d)}")


# ══════════════════════════════════════════════════════════════════
# FAILURE CASE 1: Synonym blind spot
# Metrics see surface tokens; synonyms are invisible to BLEU/ROUGE
# ══════════════════════════════════════════════════════════════════
evaluate_pair(
    reference = "Scientists have discovered a new planet outside our solar system.",
    hyp_A     = "Researchers found an exoplanet beyond our solar system.",   # ✅ human: good
    hyp_B     = "Scientists have discovered a new planet outside our solar system very quickly.",  # ❌ human: verbatim + hallucination
    label_A   = "Synonym",
    label_B   = "Verbose",
    case_title= "SYNONYM BLINDNESS — BLEU/ROUGE reward verbatim over paraphrase"
)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FAILURE CASE 2: Repetition exploit
# Repeating high-precision n-grams inflates BLEU (pre-clipping fix)
# Even with clipping, repeated words can game ROUGE-1 recall
# ══════════════════════════════════════════════════════════════════
evaluate_pair(
    reference = "The economy grew by 3 percent last quarter.",
    hyp_A     = "The economy grew by 3 percent last quarter.",   # ✅ perfect
    hyp_B     = "The the the the economy the the economy the.",   # ❌ nonsense
    label_A   = "Perfect",
    label_B   = "Repetition",
    case_title= "REPETITION EXPLOIT — ROUGE-1 recall can be gamed by repeating key words"
)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FAILURE CASE 3: Word order insensitivity
# BLEU n-grams can't fully capture that word order changes meaning
# ══════════════════════════════════════════════════════════════════
evaluate_pair(
    reference = "The dog bit the man.",
    hyp_A     = "The man bit the dog.",    # ❌ opposite meaning!
    hyp_B     = "A canine attacked a person.",  # ✅ correct meaning, different words
    label_A   = "OrderSwap",
    label_B   = "Synonym",
    case_title= "WORD ORDER INSENSITIVITY — 'Dog bit man' ≡ 'Man bit dog' to n-gram metrics"
)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FAILURE CASE 4: BERTScore rewards fluent-sounding nonsense
# High-quality fluent text shares embedding space with reference
# even when factually wrong
# ══════════════════════════════════════════════════════════════════
evaluate_pair(
    reference = "Marie Curie won the Nobel Prize in Physics in 1903.",
    hyp_A     = "Marie Curie was awarded the Nobel Prize for Chemistry.",   # ❌ wrong subject
    hyp_B     = "The Nobel Prize in Physics 1903 went to Marie Curie.",      # ✅ correct
    label_A   = "WrongFact",
    label_B   = "Correct",
    case_title= "BERT FLUENCY BIAS — semantically close but factually wrong"
)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FAILURE CASE 5: Length preference — BLEU brevity penalty
# ROUGE has no length penalty; long outputs get free recall points
# ══════════════════════════════════════════════════════════════════
evaluate_pair(
    reference = "It rained heavily.",
    hyp_A     = "It rained heavily.",       # ✅ perfect
    hyp_B     = ("It rained heavily and there was also thunder and lightning and the streets "
                 "flooded and people stayed indoors and cars got stuck in the water."),  # ❌ padded
    label_A   = "Perfect",
    label_B   = "Padded",
    case_title= "LENGTH INFLATION — ROUGE recall rewards padding; BLEU penalises only short"
)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FAILURE CASE 6: Multi-reference gaming
# With many references, any output can find high n-gram overlap
# ══════════════════════════════════════════════════════════════════
refs_many = [
    "The stock market declined yesterday.",
    "Markets fell sharply on Monday.",
    "Investors saw losses across the board.",
    "Wall Street had a bad day.",
    "Equities dropped amid uncertainty.",
]
hyp_garbage = "The sharply across Wall Street uncertainty board Monday."
hyp_good    = "Stock prices dropped significantly yesterday."

print("\n" + "="*70)
print("  FAILURE CASE: MULTI-REFERENCE GAMING")
print("  When references are diverse, garbage text borrows n-grams from each")
print("="*70)
for label, hyp in [("Garbage", hyp_garbage), ("Good", hyp_good)]:
    b  = bleu_score(hyp, refs_many)['bleu']
    r1 = max(rouge_all(hyp, r)['rouge1']['f1'] for r in refs_many)  # best-ref ROUGE
    print(f"  {label:10s} → BLEU(multi-ref)={b:.3f}   ROUGE-1(best-ref)={r1:.3f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FAILURE CASE 7: Negation blindness
# "not safe" vs "safe" — BLEU/ROUGE score them almost identically!
# ══════════════════════════════════════════════════════════════════
evaluate_pair(
    reference = "The drug is safe for children under 12.",
    hyp_A     = "The drug is not safe for children under 12.",   # ❌ opposite!
    hyp_B     = "This medication is appropriate for paediatric patients up to age 12.",  # ✅
    label_A   = "Negated",
    label_B   = "Synonym",
    case_title= "NEGATION BLINDNESS — adding 'not' barely changes BLEU/ROUGE"
)

---
## PART 5 · Systematic Comparison Dashboard

In [ ]:
# Systematic dashboard: all failure cases side-by-side

FAILURE_CASES = [
    {
        'title': '1. Synonym Blindness',
        'ref': "Scientists discovered a new planet outside our solar system.",
        'good': "Researchers found an exoplanet beyond our solar system.",
        'bad': "Scientists have discovered a new planet outside our solar system today immediately.",
        'human_winner': 'good',
    },
    {
        'title': '2. Word Order',
        'ref': "The dog bit the man.",
        'good': "A canine attacked a person.",
        'bad': "The man bit the dog.",
        'human_winner': 'good',
    },
    {
        'title': '3. Repetition',
        'ref': "The economy grew last quarter.",
        'good': "Economic expansion occurred in the recent quarter.",
        'bad': "The the the economy the economy the the the quarter.",
        'human_winner': 'good',
    },
    {
        'title': '4. Negation',
        'ref': "The drug is safe for children.",
        'good': "This medication is appropriate for paediatric patients.",
        'bad': "The drug is not safe for children.",
        'human_winner': 'good',
    },
    {
        'title': '5. Padding',
        'ref': "It rained heavily.",
        'good': "There was heavy rainfall.",
        'bad': "It rained heavily and also there was thunder and lightning and floods.",
        'human_winner': 'good',
    },
    {
        'title': '6. Factual Error',
        'ref': "Marie Curie won the Nobel Prize in Physics in 1903.",
        'good': "Marie Curie received the 1903 Nobel Prize in Physics.",
        'bad': "Marie Curie won the Nobel Prize for Chemistry in 1903.",
        'human_winner': 'good',
    },
]


def score_all(hyp, ref):
    b  = bleu_score(hyp, [ref])['bleu']
    r1 = rouge_all(hyp, ref)['rouge1']['f1']
    rl = rouge_all(hyp, ref)['rougeL']['f1']
    bs = scorer.score(hyp, ref)['f1']
    return np.array([b, r1, rl, bs])


metric_names = ['BLEU', 'ROUGE-1', 'ROUGE-L', 'BERTScore']
n_cases  = len(FAILURE_CASES)
n_metrics= len(metric_names)

good_scores = np.zeros((n_cases, n_metrics))
bad_scores  = np.zeros((n_cases, n_metrics))

for i, case in enumerate(FAILURE_CASES):
    good_scores[i] = score_all(case['good'], case['ref'])
    bad_scores[i]  = score_all(case['bad'],  case['ref'])

# metric_winner[i, j] = True if metric j correctly ranks good > bad
metric_correct = good_scores > bad_scores

fig, axes = plt.subplots(n_cases, 1, figsize=(14, n_cases * 2.5))
colors_good = '#2ecc71'
colors_bad  = '#e74c3c'

for i, (ax, case) in enumerate(zip(axes, FAILURE_CASES)):
    x = np.arange(n_metrics)
    bars_good = ax.bar(x - 0.2, good_scores[i], 0.35, label='Good (human✓)',
                       color=colors_good, alpha=0.8)
    bars_bad  = ax.bar(x + 0.2, bad_scores[i],  0.35, label='Bad  (human✗)',
                       color=colors_bad,  alpha=0.8)

    # Mark incorrect rankings
    for j in range(n_metrics):
        if not metric_correct[i, j]:
            ax.annotate('❌', xy=(j, max(good_scores[i,j], bad_scores[i,j]) + 0.03),
                        ha='center', fontsize=14)
        else:
            ax.annotate('✅', xy=(j, max(good_scores[i,j], bad_scores[i,j]) + 0.03),
                        ha='center', fontsize=14)

    ax.set_title(case['title'], fontweight='bold', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names)
    ax.set_ylim(0, 1.25)
    ax.set_ylabel('Score')
    if i == 0:
        ax.legend(loc='upper right', fontsize=9)

plt.suptitle('Metric vs. Human Judgment: ✅ = metric agrees with human | ❌ = metric fails',
             fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig('/tmp/metric_failures.png', dpi=120, bbox_inches='tight')
plt.show()

# Summary
print("\nMetric accuracy across failure cases:")
for j, m in enumerate(metric_names):
    acc = metric_correct[:, j].mean()
    print(f"  {m:12s}: {int(metric_correct[:,j].sum())}/{n_cases} correct  ({acc*100:.0f}%)")

---
## PART 6 · Correlation Analysis — When Metrics Agree and Disagree

In [ ]:
# Generate a larger sample of hypothesis/reference pairs to study metric correlations

CORPUS = [
    # (reference, hypothesis, human_score 0-1)
    ("The stock market rose today.",            "Markets gained ground on Monday.",                    0.85),
    ("The stock market rose today.",            "The stock market rose today.",                        1.00),
    ("The stock market rose today.",            "The stock market fell today.",                        0.05),
    ("The stock market rose today.",            "Equities advanced in trading.",                       0.80),
    ("The stock market rose today.",            "The weather was nice.",                               0.00),
    ("Scientists found water on Mars.",         "Researchers discovered water on the Red Planet.",     0.90),
    ("Scientists found water on Mars.",         "Scientists found water on Mars yesterday quickly.",   0.50),
    ("Scientists found water on Mars.",         "Water was not found on Mars by scientists.",          0.05),
    ("Scientists found water on Mars.",         "The planet Mars may have ice.",                       0.55),
    ("Scientists found water on Mars.",         "Aliens live on Mars.",                                0.00),
    ("She sang beautifully at the concert.",    "Her singing at the event was wonderful.",             0.88),
    ("She sang beautifully at the concert.",    "She sang beautifully at the concert tonight.",        0.70),
    ("She sang beautifully at the concert.",    "The concert was terrible.",                           0.00),
    ("She sang beautifully at the concert.",    "She performed well on stage.",                        0.80),
    ("The bridge collapsed after the flood.",   "The flood caused the bridge to collapse.",            0.92),
    ("The bridge collapsed after the flood.",   "The bridge collapsed after the flood.",               1.00),
    ("The bridge collapsed after the flood.",   "Flooding led to bridge failure.",                     0.85),
    ("The bridge collapsed after the flood.",   "The bridge was fine after the storm.",                0.05),
    ("The bridge collapsed after the flood.",   "Bridges are important infrastructure.",               0.10),
]

results_df = []
for ref, hyp, human in CORPUS:
    b  = bleu_score(hyp, [ref])['bleu']
    r1 = rouge_all(hyp, ref)['rouge1']['f1']
    rl = rouge_all(hyp, ref)['rougeL']['f1']
    bs = scorer.score(hyp, ref)['f1']
    results_df.append({'bleu': b, 'rouge1': r1, 'rougeL': rl, 'bertscore': bs, 'human': human})

import numpy as np

keys = ['bleu', 'rouge1', 'rougeL', 'bertscore', 'human']
data = {k: np.array([r[k] for r in results_df]) for k in keys}

print("Pearson correlation with human scores:")
for m in ['bleu', 'rouge1', 'rougeL', 'bertscore']:
    corr = np.corrcoef(data[m], data['human'])[0, 1]
    print(f"  {m:12s}: r = {corr:.4f}")

# Scatter plot
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
metrics_plot = [('bleu', 'BLEU'), ('rouge1', 'ROUGE-1'), ('rougeL', 'ROUGE-L'), ('bertscore', 'BERTScore')]

for ax, (m_key, m_name) in zip(axes, metrics_plot):
    ax.scatter(data[m_key], data['human'], alpha=0.7, s=60,
               color='steelblue', edgecolors='white', linewidths=0.5)
    # Regression line
    z = np.polyfit(data[m_key], data['human'], 1)
    p = np.poly1d(z)
    xs = np.linspace(0, 1, 100)
    ax.plot(xs, p(xs), 'r--', alpha=0.5)
    corr = np.corrcoef(data[m_key], data['human'])[0, 1]
    ax.set_title(f'{m_name}\nr={corr:.3f}', fontweight='bold')
    ax.set_xlabel('Metric Score')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.10)

axes[0].set_ylabel('Human Score')
plt.suptitle('Metric vs Human Score Correlation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## PART 7 · IDF Weighting in BERTScore

In [ ]:
# Demonstrate why IDF weighting helps in BERTScore
# Stop words (the, a, is) dominate without IDF; IDF down-weights them

CORPUS_FOR_IDF = [
    "The cat sat on the mat",
    "The dog lay under the table",
    "A bird flew over the house",
    "The scientist discovered a new element",
    "A child played in the garden",
]

def compute_idf(corpus: List[str]) -> Dict[str, float]:
    """Compute IDF over a small corpus."""
    N = len(corpus)
    df: Dict[str, int] = {}
    for doc in corpus:
        for tok in set(tokenize(doc)):
            df[tok] = df.get(tok, 0) + 1
    return {tok: math.log((N + 1) / (cnt + 1)) + 1 for tok, cnt in df.items()}

idf = compute_idf(CORPUS_FOR_IDF)

# Compare BERTScore with and without IDF for a tricky pair
ref_s = "The cat sat on the mat"
hyp_s = "A feline rested upon a rug"  # semantically good, surface-different

hyp_tokens = tokenize(hyp_s)
ref_tokens = tokenize(ref_s)

hyp_emb = scorer.embed_tokens(hyp_tokens)
ref_emb = scorer.embed_tokens(ref_tokens)

idf_hyp = np.array([idf.get(t, 1.0) for t in hyp_tokens])
idf_ref = np.array([idf.get(t, 1.0) for t in ref_tokens])

r_no_idf  = bertscore_from_embeddings(hyp_emb, ref_emb)
r_idf     = bertscore_from_embeddings(hyp_emb, ref_emb, idf_hyp, idf_ref)

print(f"Reference : {ref_s}")
print(f"Hypothesis: {hyp_s}")
print()
print(f"Without IDF: P={r_no_idf['precision']:.4f}  R={r_no_idf['recall']:.4f}  F1={r_no_idf['f1']:.4f}")
print(f"With IDF:    P={r_idf['precision']:.4f}    R={r_idf['recall']:.4f}    F1={r_idf['f1']:.4f}")

# Show IDF values
print("\nIDF weights for hypothesis tokens:")
for tok in hyp_tokens:
    print(f"  '{tok}': {idf.get(tok, 1.0):.3f}")
print("\n(Stop words like 'the', 'a' have low IDF → down-weighted)")

---
## PART 8 · Grand Summary & Practical Recommendations

In [ ]:
# Grand summary heatmap

failure_types = [
    'Synonym blindness',
    'Word order insensitivity',
    'Repetition exploit',
    'Negation blindness',
    'Length inflation',
    'Factual errors',
    'Multi-ref gaming',
    'Discourse coherence',
    'Hallucinations',
    'Cultural nuance',
]

metric_names_full = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERTScore']

# 0 = fails badly, 0.5 = partial, 1 = handles well
# Based on the empirical results from our experiments + literature
sensitivity_matrix = np.array([
    # BLEU  R1    R2    RL    BERT
    [0.1,  0.1,  0.2,  0.2,  0.8],   # Synonym blindness
    [0.4,  0.4,  0.5,  0.7,  0.6],   # Word order
    [0.7,  0.3,  0.6,  0.6,  0.7],   # Repetition (BLEU clips)
    [0.2,  0.2,  0.2,  0.2,  0.3],   # Negation (all fail!)
    [0.5,  0.2,  0.3,  0.4,  0.5],   # Length inflation
    [0.2,  0.2,  0.2,  0.2,  0.4],   # Factual errors
    [0.3,  0.3,  0.4,  0.4,  0.5],   # Multi-ref gaming
    [0.2,  0.2,  0.3,  0.4,  0.5],   # Discourse coherence
    [0.2,  0.2,  0.2,  0.2,  0.4],   # Hallucinations
    [0.1,  0.1,  0.1,  0.1,  0.3],   # Cultural nuance
])

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(sensitivity_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

ax.set_xticks(range(len(metric_names_full)))
ax.set_xticklabels(metric_names_full, fontsize=11, fontweight='bold')
ax.set_yticks(range(len(failure_types)))
ax.set_yticklabels(failure_types, fontsize=10)
ax.set_title('Metric Sensitivity to Failure Types\n(green = handles well, red = fails)',
             fontsize=13, fontweight='bold')

for i in range(len(failure_types)):
    for j in range(len(metric_names_full)):
        v = sensitivity_matrix[i, j]
        label = '✓' if v >= 0.7 else ('~' if v >= 0.4 else '✗')
        ax.text(j, i, f'{label}\n{v:.1f}', ha='center', va='center', fontsize=8,
                color='black', fontweight='bold')

plt.colorbar(im, ax=ax, label='Robustness (0=fails, 1=handles well)')
plt.tight_layout()
plt.show()

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║         PRACTICAL RECOMMENDATIONS — When to use what                ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  BLEU                                                                ║
║  ✅ Use for: MT benchmarking with multiple references (WMT etc.)     ║
║  ✅ Fast, reproducible, standard baseline                            ║
║  ❌ Avoid for: single-reference summarization, dialog, creative text ║
║  ❌ Never use sentence-level BLEU without smoothing                  ║
║                                                                      ║
║  ROUGE                                                               ║
║  ✅ Use for: summarization (ROUGE-2 & ROUGE-L both)                  ║
║  ✅ ROUGE-L captures structure better than ROUGE-N for long text     ║
║  ❌ Avoid for: MT, factuality evaluation                             ║
║  ❌ Long outputs game recall; always report P, R, F1 separately      ║
║                                                                      ║
║  BERTScore                                                           ║
║  ✅ Use for: tasks with rich synonym/paraphrase space                ║
║  ✅ Better correlates with human judgment on most tasks              ║
║  ❌ Expensive; biased toward in-domain fluency                       ║
║  ❌ Does NOT detect negation, factual errors, hallucinations         ║
║                                                                      ║
║  GOLDEN RULE                                                         ║
║  → Never rely on a single metric                                     ║
║  → Always combine with human evaluation or factuality checks         ║
║  → For LLM eval: use LLM-as-judge + at least one reference metric   ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

---
## PART 9 · Bonus — Implement Smoothed Corpus-Level BLEU

In [ ]:
# Corpus-level BLEU: accumulate counts across all sentences, then compute once
# This is how WMT evaluates translation systems — avoids sentence-level instability

def corpus_bleu(
    hypotheses: List[str],
    references_list: List[List[str]],   # outer = sentence idx, inner = reference variants
    max_n: int = 4,
) -> Dict:
    """
    Corpus-level BLEU.
    hypotheses       : list of hypothesis strings
    references_list  : list of lists — each entry is one or more references for that sentence
    """
    clipped_counts   = [0] * max_n
    total_counts     = [0] * max_n
    hyp_len_total    = 0
    ref_len_total    = 0

    for hyp_str, refs in zip(hypotheses, references_list):
        hyp_tokens    = tokenize(hyp_str)
        ref_tokens_list = [tokenize(r) for r in refs]

        hyp_len_total += len(hyp_tokens)
        ref_len_total += closest_reference_length(len(hyp_tokens), ref_tokens_list)

        for n in range(1, max_n + 1):
            num, den = modified_precision(hyp_tokens, ref_tokens_list, n)
            clipped_counts[n-1] += num
            total_counts[n-1]   += den

    precisions = [
        clipped_counts[n] / total_counts[n] if total_counts[n] else 0.0
        for n in range(max_n)
    ]

    bp = brevity_penalty(hyp_len_total, ref_len_total)
    weights = [1/max_n] * max_n

    log_avg = sum(
        w * math.log(p) if p > 0 else float('-inf')
        for w, p in zip(weights, precisions)
    )
    score = bp * math.exp(log_avg) if log_avg != float('-inf') else 0.0

    return {'bleu': score, 'precisions': precisions, 'bp': bp,
            'hyp_len': hyp_len_total, 'ref_len': ref_len_total}


# Demo
hyps = [
    "The cat sat on the mat",
    "The dog ran in the park",
    "She sang a lovely song",
]
refs = [
    ["The cat sat on the mat", "A cat sat on a mat"],
    ["The dog ran through the park"],
    ["She sang a beautiful song", "She performed a wonderful song"],
]

result = corpus_bleu(hyps, refs)
print(f"Corpus BLEU: {result['bleu']:.4f}")
print(f"Precisions : {[f'{p:.3f}' for p in result['precisions']]}")
print(f"BP         : {result['bp']:.4f}")

---

## Summary

| What we built | Key insight |
|--------------|------------|
| `bleu_score()` with clipping, BP, smoothing | Zero in any $p_n$ kills the score; always smooth for sentence-level |
| `rouge_n/l/s()` with LCS + skip-bigrams | Recall-biased; ROUGE-L captures order better than ROUGE-2 |
| `BERTScorer` with IDF weighting | Captures semantics but misses negation, facts, hallucinations |
| 7 deliberate failure cases | **Negation** breaks all three; **synonyms** break only BLEU/ROUGE; **fluency bias** tricks BERTScore |

### The meta-lesson

> **Metrics measure proxies, not meaning.** They are useful for fast, cheap, reproducible ranking — but they are not a substitute for human evaluation, especially when factuality, negation, or nuance matters.

---
*Implemented from scratch — no sacrebleu / evaluate / rouge_score libraries used.*